<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_2_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 - Image Classification

In This laboration you will explore supervised learning for image classification, in a very interactive sense. The subject of the laboration is you and your environment. You will start by collecting training data and learn a computer vision model, a deep neural network in this case, to recognize you.
Then you will investigate several different aspects that affects how well machine learning for vision works, and what you can do to overcome some of these, as you explore several different challenges.

### **A note on privacy**
This notebook is running on Google cloud infrastructure.
You will be tasked to collect a lot of images using your webcam, which will contain images of you, possibly of others and of different objects that might (or might not) be sensitive. **However, no images that you capture will leave your computer.** We have built this laboration such that all machine learning and auxiliary functionality is running as JavaScript in your browser, including image capture from the camera, data set collection, neural network training and image classification using the trained machine learning model.

### **Authors**
Mathias Ahlgren<br />
Mattias Tiger, mattias.tiger@liu.se

### **License**
CC BY-NC-SA 4.0 <br />
https://creativecommons.org/licenses/by-nc-sa/4.0/

The lab (Notebook: code and instructions) is free to use, share, change and to make your own version suitable for your own students. The only restriction to this version and any new version is that 1) the same license (CC BY-NC-SA 4.0) is applied, 2) that the above mentioned authors are referred to as the authors of the original version and 3) that it cannot be used in a setting where a person or business has to pay to get access to, or in order to use, the version. I.e. full permission is given to use the lab, and subsequent versions, in schools or study circles, even running on paid-for computational resources (e.g. paid version of Google colab).
<hr />

# Image Classificaiton - Lab 2.0 - Setup of Machine Learning web app

First, we have to install a web server to host our web app:


In [ ]:
!pip install flask


<hr />
In order to get the web camera working in our colab setup, we have to implement the following work-around:

In [ ]:
# This has to be done in order for the Iframe to be able to request the camera,
# I dont know why but I think its due to hierarchical permissions in html
from IPython.display import display, Javascript

display(Javascript("""
(async () => {
    try {
        await navigator.mediaDevices.getUserMedia({ video: true });
        console.log('Camera permission granted in parent frame.');
    } catch (err) {
        console.error('Error accessing the webcam in parent frame: ' + err);
        alert('Error accessing the webcam in parent frame: ' + err);
    }
})();
"""))

<IPython.core.display.Javascript object>

<hr />
Then we have to implement the web app:

In [1]:
# What can I do to pre-process images, so it doesnt become overfitted on the background instead of the subject
# Maybe pre-process with segmentation of subject vs background?

from flask import Flask
from IPython.display import Javascript, display, IFrame
import threading

app = Flask(__name__)

@app.route('/')
def index():
    return     '''<!DOCTYPE html>
<html lang="en">
<head>
    <!-- Meta and Title -->
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <!-- Styles -->
    <style>
    /* General Styles */
body {
    font-family: Arial, sans-serif;
    background-color: #1e1e2f; /* Dark blue-gray */
    color: #f5f5f5; /* Off-white text */
    margin: 0;
    padding: 20px;
    text-align: center;

    overflow-x: hidden;
    display: flex;
    align-items: center;
    justify-content: center;
}

h1, h2 {
    margin-top: 20px;
    color: #ffffff; /* Purple */
    font-size: 16px;
}

.container {
    max-width: auto;
    margin: auto;
}

/* Main Content Layout */
.main-content {
    display: flex;
    justify-content: space-between;
    margin-top: 20px;

    display: grid;
    grid-template-columns: repeat(3, 1fr);

    row-gap: 10px;
}

.column {
    padding: 20px;
    background: linear-gradient(135deg, #2a2a40, #323250); /* Dark gradient */
    border-radius: 10px;
    max-height: 600px;
    overflow-y: scroll;
}

.middle-column {
    background: linear-gradient(135deg, #2b2d42, #353554); /* Slightly lighter gradient */
    margin: 0 10px;
}

/* Webcam and Control Styles */
#webcam-container, #controls, #prediction-container {
    margin-top: 20px;
}

#webcam-container {
    position: relative;
    display: inline-block;
}

#webcam-container video, #webcam-container canvas {
    border: 2px solid #55415c; /* Purple border */
    border-radius: 10px;
}

button {
    padding: 5px 8px;
    background-color: #9b59b6; /* Purple */
    color: #fff;
    border: none;
    border-radius: 5px;
    cursor: pointer;
    margin: 5px;
    font-size: 16px;
}

button:hover {
    background-color: #ab69c6; /* Lighter purple */
}

input[type="text"], input[type="number"] {
    padding: 8px;
    font-size: 12px;
    width: 80%;
    border: 2px solid #55415c;
    border-radius: 5px;
    margin: 5px auto;
    color: #f5f5f5;
    background-color: #3d3d59; /* Darker background for inputs */
}

/* Class Container Styles */
.class-container {
    border: 1px solid #55415c; /* Purple */
    padding: 10px;
    margin: 10px;
    border-radius: 5px;
    text-align: left;
}

.class-container h3 {
    margin-top: 0;
    color: #f5f5f5; /* Off-white */
}

/* Flexbox for aligning class heading and capture button on the same row */
.class-container .row {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 10px;
}

/* Hover effects for images in the image container */
.image-container {
    display: flex;
    flex-wrap: wrap;
    margin-top: 10px;
    position: relative;
}

.image-container canvas {
    margin: 5px;
    border: 1px solid #9b59b6;
    width: 50px;
    height: 50px;
    position: relative;
}

/* Remove Image Cross Button */
.remove-image {
    position: absolute;
    top: -8px;
    right: -8px;
    background-color: #ff4081; /* Coral */
    border: none;
    color: white;
    font-size: 12px;
    border-radius: 50%;
    cursor: pointer;
    width: 20px;
    height: 20px;
    display: none;
    align-items: center;
    justify-content: center;
}

/* Show the remove button when hovering over the image container */
.thumbnail-wrapper:hover .remove-image {
    display: flex;
}

.thumbnail-wrapper {
    position: relative;
    display: inline-block;
}

/* Hidden Class for Overflow Images */
.hidden {
    display: none;
}

/* Image Preview */
#image-preview {
    border: 2px solid #9b59b6;
    border-radius: 10px;
    margin-top: 20px;
    width: 100%;
}

/* Training Status */
#train-status {
    margin-top: 20px;
    font-size: 18px;
    color: #ff4081; /* Coral */
}

/* Prediction Display */
#prediction-bars {
    display: flex;
    flex-direction: column;
    margin-top: 10px;
}

.prediction-bar {
    display: flex;
    align-items: center;
    margin: 5px 0;
}

.prediction-bar-label {
    flex-shrink: 0;
    width: 150px;
    text-align: left;
    color: #fff;
}

.prediction-bar-fill {
    flex-grow: 1;
    height: 25px;
    background-color: white;
    border-radius: 5px;
    margin-left: 10px;
}

.prediction-bar-fill .bar {
    height: 100%;
    background-color: #9b59b6; /* Purple bar */
    border-radius: 5px;
}

/* Button for showing more images */
.show-more-btn {
    margin-top: 10px;
    background-color: #9b59b6; /* Purple button */
    color: white;
    border: none;
    border-radius: 5px;
    padding: 5px 10px;
    cursor: pointer;
}

.class-input-group {
    display: flex;
    align-items: row;
    justify-content: center;
}

canvas {
    border: 2px solid #9b59b6;
    border-radius: 5px;
    margin-top: 10px;
}
</style>
</head>
<body>
    <div class="container">
        <div class="main-content">
            <div class="column left-column">
                <h2>Adjust Hyperparameters</h2>
                <form id="hyperparameters-form">
                    <div>
                        <label for="epochs">Epochs:</label>
                        <input type="number" id="epochs" name="epochs" value="10" min="1">
                    </div>

                    <div>
                        <label for="batchSize">Batch Size:</label>
                        <input type="number" id="batchSize" name="batchSize" value="5" min="1">
                    </div>

                    <div>
                        <label for="learningRate">Learning Rate:</label>
                        <input type="number" id="learningRate" name="learningRate" value="0.001" step="0.001" min="0">
                    </div>

                    <div>
                        <label for="units">Hidden Layer Neurons:</label>
                        <input type="number" id="units" name="units" value="100" min="1">
                    </div>

                    <button type="button" onclick="trainModel()">Train Model</button>
                    <div id="train-status"></div>
                </form>

                <canvas id="loss-chart" width="400" height="200"></canvas>
                <canvas id="accuracy-chart" width="400" height="200"></canvas>
            </div>

            <!-- Middle Column (Webcam and Predictions) -->
            <div class="column middle-column">
                <h2>Webcam and Predictions</h2>

                <div id="webcam-container"></div>

                <div id="controls">
                    <button id="webcam-toggle" onclick="toggleWebcam()">Pause Webcam</button>
                    <button id="start-webcam" onclick="startWebcam()">Start Webcam</button>
                </div>

                <div id="prediction-container">
                    <div id="prediction-bars"></div>
                </div>
            </div>

            <!-- Right Column (Classes and Capturing) -->
            <div class="column right-column">
                <h2>Add Classes and Images</h2>

                <div class="class-input-group">
                    <input type="text" style="width: 80%" id="class-label-input" placeholder="Enter class label" />
                    <button type="button" styele="height: 70%" onclick="addClass()">+</button>
                </div>

                <div id="classes-container"></div>
            </div>
        </div>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.21.0/dist/tf.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/@tensorflow-models/mobilenet@2.1.0/dist/mobilenet.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

    <script type="text/javascript">
        let model;
        let mobilenetModule;
        let webcamElement;
        let canvasElement;
        let isPredicting = false;
        let trainingData = {};
        let classNames = [];
        let numClasses = 0;
        let webcamPaused = false;
        let captureInterval; // For continuous capture
        let lossChart, accuracyChart;
        let lastLogTime = 0;

        window.addEventListener('load', async () => {
            document.getElementById("webcam-toggle").style.display = 'none';
            await loadMobilenetModel();
            createCharts(); // Create charts on load
        });

        async function loadMobilenetModel() {
            mobilenetModule = await mobilenet.load();
            console.log('MobileNet model loaded.');
        }

        async function startWebcam() {
            try {
                console.log('Starting webcam...');
                await initWebcam();
                document.getElementById("start-webcam").style.display = 'none';
                document.getElementById("webcam-toggle").style.display = 'inline-block';
            } catch (error) {
                console.error("Error setting up webcam:", error);
                document.getElementById("start-webcam").style.display = 'inline-block';
                document.getElementById("webcam-toggle").style.display = 'none';
                alert('Failed to load webcam. Error: ' + error.message);
            }
        }

        async function initWebcam() {
            console.log('Initializing webcam...');
            webcamElement = document.createElement('video');
            webcamElement.setAttribute('autoplay', '');
            webcamElement.setAttribute('playsinline', '');
            webcamElement.width = 300;
            webcamElement.height = 300;
            webcamElement.style.transform = 'scaleX(-1)';
            document.getElementById("webcam-container").appendChild(webcamElement);

            const stream = await navigator.mediaDevices.getUserMedia({ video: true, flipHorizontal: true });
            webcamElement.srcObject = stream;

            await new Promise((resolve) => {
                webcamElement.onloadedmetadata = () => {
                    resolve();
                };
            });

            webcamElement.play();

            canvasElement = document.createElement('canvas');
            canvasElement.width = 224;
            canvasElement.height = 224;
            canvasElement.style.display = 'none';
            document.getElementById("webcam-container").appendChild(canvasElement);

            window.requestAnimationFrame(loop);
        }

        async function loop() {
            if (!webcamPaused) {
                captureFrame();
                if (isPredicting) {
                    await predict();
                }
            }
            window.requestAnimationFrame(loop);
        }

        function captureFrame() {
            tf.tidy(() => {
                const context = canvasElement.getContext('2d');
                context.drawImage(webcamElement, 0, 0, canvasElement.width, canvasElement.height);
            });
        }

        function toggleWebcam() {
            if (webcamPaused) {
                webcamPaused = false;
                webcamElement.play();
                document.getElementById('webcam-toggle').innerText = "Pause Webcam";
            } else {
                webcamPaused = true;
                webcamElement.pause();
                document.getElementById('webcam-toggle').innerText = "Resume Webcam";
            }
        }

        function addClass() {
            const className = document.getElementById('class-label-input').value.trim();
            if (className && !classNames.includes(className)) {
                classNames.push(className);
                trainingData[className] = {
                    images: [],
                    labels: []
                };
                numClasses = classNames.length;

                const classContainer = document.createElement('div');
                classContainer.className = 'class-container';

                const row = document.createElement('div');
                row.className = 'row';

                const classHeading = document.createElement('h3');
                classHeading.innerText = `Class: ${className}`;
                row.appendChild(classHeading);

                const imageCounter = document.createElement('span');
                imageCounter.id = `image-counter-${className}`;
                imageCounter.innerText = `0`;
                row.appendChild(imageCounter);

                const classButton = document.createElement('button');
                classButton.innerText = `Capture`;
                classButton.onmousedown = () => startContinuousCapture(className);
                classButton.onmouseup = () => stopContinuousCapture();
                classButton.onmouseleave = () => stopContinuousCapture();
                row.appendChild(classButton);

                const removeClassButton = document.createElement('button');
                removeClassButton.innerText = "X";
                removeClassButton.style.backgroundColor = "#ff4081";
                removeClassButton.onclick = () => removeClass(classContainer, className);
                row.appendChild(removeClassButton);

                classContainer.appendChild(row);

                const imageContainer = document.createElement('div');
                imageContainer.className = 'image-container';
                imageContainer.id = `images-${className}`;

                classContainer.appendChild(imageContainer);
                document.getElementById('classes-container').appendChild(classContainer);

                document.getElementById('class-label-input').value = '';
            } else {
                alert('Please enter a unique class name.');
            }
        }

        function startContinuousCapture(className) {
            captureImage(className);
            captureInterval = setInterval(() => captureImage(className), 100);
        }

        function stopContinuousCapture() {
            clearInterval(captureInterval);
        }

        function captureImage(label) {
            const img = tf.browser.fromPixels(canvasElement);

            trainingData[label].images.push(img);
            trainingData[label].labels.push(classNames.indexOf(label));

            const imageContainer = document.getElementById(`images-${label}`);
            const thumbnailWrapper = document.createElement('div');
            thumbnailWrapper.className = 'thumbnail-wrapper';

            const thumbnail = document.createElement('canvas');
            thumbnail.width = 50;
            thumbnail.height = 50;
            const thumbnailCtx = thumbnail.getContext('2d');
            thumbnailCtx.drawImage(canvasElement, 0, 0, 50, 50);

            const removeImageBtn = document.createElement('button');
            removeImageBtn.innerText = "X";
            removeImageBtn.className = 'remove-image';
            removeImageBtn.onclick = () => removeImage(thumbnailWrapper, label, trainingData[label].images.length - 1);

            thumbnailWrapper.appendChild(thumbnail);
            thumbnailWrapper.appendChild(removeImageBtn);

            const showMoreBtn = document.getElementById(`show-more-btn-${label}`);
            if (showMoreBtn) {
                imageContainer.insertBefore(thumbnailWrapper, showMoreBtn);
            } else {
                imageContainer.appendChild(thumbnailWrapper);
            }

            updateImageDisplay(imageContainer, label);
            updateImageCounter(label);
        }

        function updateImageCounter(label) {
            const imageCounter = document.getElementById(`image-counter-${label}`);
            imageCounter.innerText = `${trainingData[label].images.length}`;
        }

      function removeImage(thumbnailWrapper, label, imageIndex) {
          trainingData[label].images.splice(imageIndex, 1);
          trainingData[label].labels.splice(imageIndex, 1);

          thumbnailWrapper.remove();

          const imageContainer = document.getElementById(`images-${label}`);
          const showMoreBtn = document.getElementById(`show-more-btn-${label}`);

          updateImageDisplay(imageContainer, label);

          if (showMoreBtn && showMoreBtn.innerText === 'Show Less') {
              toggleImages(imageContainer, showMoreBtn);
              showMoreBtn.innerText = 'Show Less';  // Keep "Show Less" state
          }

          // Update the image counter
          updateImageCounter(label);
      }


        function removeClass(classContainer, label) {
            delete trainingData[label];
            classNames = classNames.filter(className => className !== label);
            numClasses--;

            classContainer.remove();
        }

        function updateImageDisplay(imageContainer, label) {
            const images = trainingData[label].images;
            const imageWrappers = imageContainer.querySelectorAll('.thumbnail-wrapper');

            if (images.length > 10) {
                imageWrappers.forEach((imgWrapper, index) => {
                    if (index >= 10) {
                        imgWrapper.classList.add('hidden');
                    }
                });
            }

            // Adjust "Show More" / "Show Less" button
            let showMoreBtn = document.getElementById(`show-more-btn-${label}`);
            if (images.length > 10 && !showMoreBtn) {
                showMoreBtn = document.createElement('button');
                showMoreBtn.innerText = "Show More";
                showMoreBtn.id = `show-more-btn-${label}`;
                showMoreBtn.onclick = () => toggleImages(imageContainer, showMoreBtn);
                imageContainer.appendChild(showMoreBtn);
            } else if (images.length <= 10 && showMoreBtn) {
                showMoreBtn.remove();
            }
        }

        function toggleImages(imageContainer, showMoreBtn) {
            const hiddenImages = imageContainer.querySelectorAll('.hidden');
            const allImages = imageContainer.querySelectorAll('.thumbnail-wrapper');

            if (hiddenImages.length > 0) {
                hiddenImages.forEach(img => img.classList.remove('hidden'));
                showMoreBtn.innerText = "Show Less";
            } else {
                allImages.forEach((imgWrapper, index) => {
                    if (index >= 10) {
                        imgWrapper.classList.add('hidden');
                    }
                });
                showMoreBtn.innerText = "Show More";
            }
        }

        async function trainModel() {
            document.getElementById('train-status').innerText = 'Preparing to train model...';
            await new Promise(resolve => setTimeout(resolve, 100));  // Ensure it updates the DOM before proceeding

            if (numClasses < 2) {
                alert('Please add at least two classes.');
                return;
            }

            resetCharts();

            document.getElementById('train-status').innerText = 'Training model...';

            let xs = null;
            let ys = null;
            for (const label in trainingData) {
                const images = trainingData[label].images;
                const labels = trainingData[label].labels;
                for (let i = 0; i < images.length; i++) {
                    const img = images[i].toFloat().div(tf.scalar(127)).sub(tf.scalar(1));
                    const activation = mobilenetModule.infer(img, 'global_average_pooling2d_1');

                    if (xs == null) {
                        xs = activation;
                        ys = tf.tensor1d([labels[i]]);
                    } else {
                        xs = xs.concat(activation, 0);
                        ys = ys.concat(tf.tensor1d([labels[i]]), 0);
                    }

                    img.dispose();
                }
            }

            model = tf.sequential({
                layers: [
                    tf.layers.dense({
                        inputShape: [1024],
                        units: 100,
                        activation: 'relu'
                    }),
                    tf.layers.dense({
                        units: numClasses,
                        activation: 'softmax'
                    })
                ]
            });

            const epochs = document.getElementById("epochs").value;
            const batchSize = document.getElementById("batchSize").value;
            const learningRate = document.getElementById("learningRate").value;

            model.compile({
                optimizer: tf.train.adam(parseFloat(learningRate)),
                loss: 'sparseCategoricalCrossentropy',
                metrics: ['accuracy']
            });

            await model.fit(xs, ys, {
                epochs: parseInt(epochs),
                batchSize: parseInt(batchSize),
                callbacks: {
                    onEpochEnd: (epoch, logs) => {
                        const now = Date.now();
                        if (now - lastLogTime > 300) {  // 300ms interval for updates
                            document.getElementById('train-status').innerText =
                                `Epoch ${epoch + 1}: loss=${logs.loss.toFixed(4)}, accuracy=${(logs.acc * 100).toFixed(2)}%`;

                            // Update charts
                            lossChart.data.labels.push(epoch + 1);
                            lossChart.data.datasets[0].data.push(logs.loss);
                            lossChart.update();

                            accuracyChart.data.labels.push(epoch + 1);
                            accuracyChart.data.datasets[0].data.push(logs.acc * 100);
                            accuracyChart.update();

                            lastLogTime = now;
                        }
                    },
                    onTrainEnd: () => {
                        document.getElementById('train-status').innerText += 'Training complete!';
                        console.log('Training complete. Starting predictions.');
                        isPredicting = true;  // Now, predictions can start
                        window.requestAnimationFrame(loop);  // Continue webcam loop with predictions
                    }

                }
            });

            xs.dispose();
            ys.dispose();
        }

        function createCharts() {
            const ctxLoss = document.getElementById('loss-chart').getContext('2d');
            const ctxAccuracy = document.getElementById('accuracy-chart').getContext('2d');

            lossChart = new Chart(ctxLoss, {
                type: 'line',
                data: {
                    labels: [],
                    datasets: [{
                        label: 'Loss',
                        data: [],
                        fill: false,
                        borderColor: 'red'
                    }]
                }
            });

            accuracyChart = new Chart(ctxAccuracy, {
                type: 'line',
                data: {
                    labels: [],
                    datasets: [{
                        label: 'Accuracy',
                        data: [],
                        fill: false,
                        borderColor: 'blue'
                    }]
                }
            });
        }

        async function predict() {
    // Check if the prediction bars container is available in the DOM
    const predictionBarsContainer = document.getElementById('prediction-bars');

    if (!predictionBarsContainer) {
        console.error('Prediction bars container not found.');
        return;
    }

    if (webcamElement && model) {
        tf.tidy(() => {
            const img = tf.browser.fromPixels(canvasElement).toFloat().div(tf.scalar(127)).sub(tf.scalar(1));
            const activation = mobilenetModule.infer(img, 'global_average_pooling2d_1');
            const reshapedActivation = activation.reshape([1, 1024]);
            const prediction = model.predict(reshapedActivation);

            prediction.data().then(predictionsData => {
                updatePredictionBars(predictionsData);  // Call function to update bars
            });
        });
    }
}


        function updatePredictionBars(predictionsData) {
    const predictionBarsContainer = document.getElementById('prediction-bars');

    // Check if predictionBarsContainer exists
    if (!predictionBarsContainer) {
        console.error('Prediction bars container not found.');
        return;
    }

    predictionBarsContainer.innerHTML = '';  // Clear previous predictions

    for (let i = 0; i < predictionsData.length; i++) {
        const className = classNames[i];
        const probability = predictionsData[i] * 100;

        const barContainer = document.createElement('div');
        barContainer.classList.add('prediction-bar');

        const label = document.createElement('span');
        label.classList.add('prediction-bar-label');
        label.innerText = `${className}: ${probability.toFixed(2)}%`;

        const barFill = document.createElement('div');
        barFill.classList.add('prediction-bar-fill');

        const bar = document.createElement('div');
        bar.classList.add('bar');
        bar.style.width = `${probability}%`;

        barFill.appendChild(bar);
        barContainer.appendChild(label);
        barContainer.appendChild(barFill);
        predictionBarsContainer.appendChild(barContainer);
    }
}
        function resetCharts() {
        // Reset Loss Chart
        lossChart.data.labels = [];
        lossChart.data.datasets[0].data = [];
        lossChart.update();

        // Reset Accuracy Chart
        accuracyChart.data.labels = [];
        accuracyChart.data.datasets[0].data = [];
        accuracyChart.update();
    }
    </script>
</body>
</html>

'''

#display(IFrame(src="http://localhost:8888", width="100%", height=600))

def show_port(port, height=600):
    display(Javascript(f"""
    (async () => {{
        const url = await google.colab.kernel.proxyPort({port});
        const iframe = document.createElement('iframe');
        iframe.src = url;
        iframe.width = '90%';
        iframe.height = '800';
        iframe.frameBorder = 0;
        iframe.allow = 'accelerometer; autoplay; gyroscope; magnetometer; xr-spatial-tracking; clipboard-write; camera; microphone'; // Add premissions for camera and microphone.
        document.body.appendChild(iframe);
    }})();
    """))


<hr />
Finally, we can start the web app and get on with the tasks!



1.   Run the code cell below.
2.   Start the web camera (use the button in the web app: "Start Webcam")
3.   "OK" the alert and agree to give the website permission to use the webcam.
4.   Re-run the code cell below, and make sure that the webcam can now be started and that it is indeed showing your webcam image stream.



In [2]:
PORT = 8001

def run_app():
    app.run(host='0.0.0.0', port=PORT)

thread = threading.Thread(target=run_app)
thread.start()

show_port(PORT)

<IPython.core.display.Javascript object>

**Great! Now lets get on with the tasks.**

# Image Classification - Lab 2.1 - Recognising YOU

<p><b>Objective:</b> Train an image classification model to distinguish between you and the absence of you (i.e. the background). This two-class classification class get you going with the web app and help you realize that classification is only meaningful if there are more than one class to distinguish between.</p>

*(Run the code cell below and use that web app for this task.)*

###Step 1: Data Collection
<p>Use your webcam to collect 100 images for each class: You (i.e. your name) and "Background".</p>

<h3>Step 2: Train the Model</h3>
<p>Use the default hyperparameters (Epochs: 10, Batch Size: 5, Learning Rate: 0.001, Hidden Layer Neurons: 100) and train the model using your collected dataset.</p>

* <h3>Question 1.1</h3><p><b>Does it keep working well when you move around?</b></p>

* <h3>Question 1.2</h3>
<p><b>Can you improve the performance by collecting more images?</b></p>

* <h3>Question 1.3</h3>
<p><b>What problems with classification accuracy did you experience and what new training data did you collect to improve it? Did it work well?</b></p>

In [ ]:
# Start a new web app instance
PORT = 8101

def run_app():
    app.run(host='0.0.0.0', port=PORT)

thread = threading.Thread(target=run_app)
thread.start()

show_port(PORT)

<IPython.core.display.Javascript object>

# Image Classification - Lab 2.2 - Rock, Paper, Scissors

<p><b>Objective:</b> Train an image classification model to distinguish between hand gestures: Rock, Paper, and Scissors. This step helps you understand the fundamentals of image classification and the importance of training data diversity.</p>

*(Run the code cell below and use that web app for this task.)*

<h3>Step 1: Data Collection</h3>
<p>Use your webcam to collect 100 images for each class: "Rock," "Paper," and "Scissors." To make the model preform better, try to limit details in the background. If possible record your hand gestures with a white or neutral background.</p>

* <h3>Question 2.1</h3><p><b>Why is it important to collect multiple examples for each gesture? What would happen if you trained the model with only one or two examples of each gesture?</b></p>

* <h3>Question 2.2</h3><p><b>What issues did you encounter while collecting data (e.g., lighting, angle)? How might these factors affect the performance of your model?</b></p>


<h3>Step 2: Train the Model</h3>
<p>Use the default hyperparameters (Epochs: 10, Batch Size: 5, Learning Rate: 0.001, Hidden Layer Neurons: 100) and train the model using your collected dataset.</p>

* <h3>Question 2.3</h3><p><b>What is the initial accuracy of your model? How did it perform when you tested it with the live webcam? Are there any gestures it classified incorrectly?</b></p>

<h3>Step 3: Adjust Hyperparameters</h3>
<p>Adjust the learning rate, batch size, and epochs one by one, and retrain the model and add more images. Compare the results of each change.</p>

* <h3>Question 2.4</h3><p><b>Which set of hyperparameters gave the best results? What impact did changing the learning rate and batch size have on the accuracy and speed of training?</b></p>


In [ ]:
# Start a new web app instance
PORT = 8102

def run_app():
    app.run(host='0.0.0.0', port=PORT)

thread = threading.Thread(target=run_app)
thread.start()

show_port(PORT)

 * Serving Flask app '__main__'
 * Debug mode: off


<IPython.core.display.Javascript object>

Address already in use
Port 8101 is in use by another program. Either identify and stop that program, or start the server with a different port.


# Image Classification - Lab 2.3 - Explore your environment

<p><b>Objective:</b> Train a new image classification model to identify common objects around your environment.

*(Run the code cell below and use that web app for this task.)*

<h3>Step 1: Select Your Objects</h3>
<p>Choose three objects from your home (e.g., mug, book, pen) and capture 50 images of each object using your webcam. Make sure to vary the angles, lighting, and backgrounds for each object.</p>

* <h3>Question 3.1</h3>
<p><b>How does varying the background, lighting, and angles affect your dataset? Why is it important to include variations in your training data?</b></p>

<h3>Step 2: Train the Model</h3>
<p>Train your new model on the collected images of the three objects using the same hyperparameters as in Part 1.</p>

* <h3>Question 3.2</h3>
<p><b>Did your model perform better or worse than in the rock-paper-scissors task? What challenges did you face in classifying real-world objects compared to hand gestures?</b></p>

<h3>Step 3: Experiment with More Objects</h3>
<p>Choose additional objects (e.g., a cup, a shoe) and retrain the model by adding these new classes. You should have 6 classes in total now.</p>

* <h3>Question 3.3</h3>
<p><b>As you add more classes to your model, how does the accuracy change? What strategies can you use to prevent the model from confusing similar-looking objects?</b></p>

In [ ]:
# Start a new web app instance
PORT = 8103

def run_app():
    app.run(host='0.0.0.0', port=PORT)

thread = threading.Thread(target=run_app)
thread.start()

show_port(PORT)

 * Serving Flask app '__main__'
 * Debug mode: off


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'


Address already in use
Port 8102 is in use by another program. Either identify and stop that program, or start the server with a different port.


<hr />

# Image Classification - Lab 2.4 - Lab Reflection

<p><b>Reflection:</b> What have you learned from exploring different classification tasks? How do the challenges and solutions differ between classifying gestures and real-world objects? What would you do differently if you had to repeat this experiment?</p>

<p><b>Next Steps:</b> Consider how AI-based classification models could be applied in your everyday work-day. What potential projects or applications could you see being useful, and how would you go about implementing them?</p>

<p><i>Write your response here.</i></p>

 * Debug mode: off


Address already in use
Port 8103 is in use by another program. Either identify and stop that program, or start the server with a different port.
